
# Elite Dangerous Local Database
> Cached Elite Dangerous systems data

In [ ]:
#| default_exp eddb.localdb

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import sys, logging, typing, sqlite3, os, json
import pandas as pd

from typing import Any, NamedTuple
from contextlib import contextmanager
from edcompanion.core import configuration


In [ ]:
from confproxy.core import init_console_logging
from edcompanion.eddb.readers import dbfilereader, dbfile_process
init_console_logging(__name__)

2025-12-20T18:00:05+0100 INFO	26834	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| exporti
syslog = logging.getLogger(__name__)
eddb_config = configuration["EDDB"]
syslog.info(f"Loading module {__name__}, config={eddb_config}")


2025-12-20T18:00:05+0100 INFO	26834	__main__	4091137986.py	<module>	4	Loading module __main__, config=Section EDDB in /home/fenke/.config/EDTravelCompanion/settings.ini


## SQLite

### SQLiteQueryParams

In [ ]:
#| export
class SQLiteQueryParams(NamedTuple):
    as_param: typing.Callable
    append_param: typing.Callable   
    get_params: typing.Callable  


In [ ]:
#| export

def sqlite_query_params(log=None) -> SQLiteQueryParams:
    sql_params = {}

    def as_param(name:str):
        return f":{str(name)}"
   
    def append_param(name:str, value:Any):
        assert str(name) not in sql_params, f"Duplicate parameter {name}"
        assert len(sql_params) < 32766, "SQLite does not allow more then approx. 32k bound parameters"

        last_name = str(name)
        sql_params[last_name] = value
        return as_param(last_name)
    
    def get_params():
        return sql_params.copy()
    
    return SQLiteQueryParams(
        as_param=as_param,
        append_param=append_param if log is None else lambda p: log(append_param(p)),
        get_params=get_params
    )

### SQLiteConnectionInterface

In [ ]:
#| export

class SQLiteConnectionInterface(typing.NamedTuple):
    cursor: typing.Callable
    commit: typing.Callable
    close: typing.Callable
    execute: typing.Callable
    executemany: typing.Callable


In [ ]:
sys.version_info

sys.version_info(major=3, minor=11, micro=10, releaselevel='final', serial=0)

In [ ]:
#| export
def sqllite_connection_interface(
        database:str=":memory:",
    ) -> SQLiteConnectionInterface:

    vi = sys.version_info
    assert vi.major >= 3, f"Python >= 3.0 required. Found {vi.major}.{vi.minor}.{vi.micro}"

    if vi.minor > 11:   
        syslog.info("Using legacy autocommit")
        connection = sqlite3.connect(database, autocommit=sqlite3.LEGACY_TRANSACTION_CONTROL, isolation_level='DEFERRED')
    else:
        syslog.info("Using isolation_level=None")
        connection = sqlite3.connect(database, isolation_level=None)

    def close():
        syslog.info("Closing connection")
        connection.close()

    def commit():
        syslog.info("Committing on connection")
        connection.commit()

    def rollback():
        syslog.info("Rolling back connection")
        connection.rollback()
    
    def cursor():
        syslog.info("Creating cursor")
        return connection.cursor()
    
    def execute(sql:str, params:tuple|dict=()):
        return connection.execute(sql, params)
    
    def execute_many(sql:str, params:list[tuple|dict]=[]):
        return connection.executemany(sql, params)

    syslog.info("Returning connection-interface")
    return SQLiteConnectionInterface(
        cursor=cursor,
        commit=commit,
        close=close,
        execute=execute,
        executemany=execute_many
    )

### Context manager

In [ ]:
#| export

@contextmanager
def sqllite_connection(*args, **kwargs):
    syslog.info(f"Opening connection-interface to {args}")
    interface = sqllite_connection_interface(*args, **kwargs)
    try:
        yield interface
    finally:
        interface.close()

### Test

In [ ]:
with sqllite_connection() as ci:
    
    ci.execute("CREATE TABLE lang(name, first_appeared)")

    # This is the named style used with executemany():
    data = (
        {"name": "C", "year": 1972},
        {"name": "Fortran", "year": 1957},
        {"name": "Python", "year": 1991},
        {"name": "Go", "year": 2009},
    )
    ci.executemany("INSERT INTO lang VALUES(:name, :year)", data)
    for r in ci.execute("SELECT * FROM lang"):
        print(r)
        
    # This is the qmark style used in a SELECT query:
    params = (1972,)

    for r in ci.execute("SELECT * FROM lang WHERE first_appeared = :year", {'year':1957}):
        print(r)



2025-12-20T18:00:14+0100 INFO	26834	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ()
2025-12-20T18:00:14+0100 INFO	26834	__main__	3862603891.py	sqllite_connection_interface	13	Using isolation_level=None
2025-12-20T18:00:14+0100 INFO	26834	__main__	3862603891.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-20T18:00:14+0100 INFO	26834	__main__	3862603891.py	close	17	Closing connection


('C', 1972)
('Fortran', 1957)
('Python', 1991)
('Go', 2009)
('Fortran', 1957)


## ED galaxy systems database

In [ ]:
eddb_config['local_data_folder'], eddb_config['main_database']

('/home/fenke/repos/EDCompanion/data', 'eddb_systems.db')

### Systems

In [ ]:
systems_databasefile = os.path.join(eddb_config['local_data_folder'], eddb_config['main_database'])
print(systems_databasefile)

/home/fenke/repos/EDCompanion/data/eddb_systems.db


In [ ]:
with sqllite_connection(systems_databasefile) as conn:
    conn.execute(f"""
    CREATE TABLE IF NOT EXISTS systems (
        id64 BIGINT NOT NULL,
        x DOUBLE PRECISION  NOT NULL,
        y DOUBLE PRECISION  NOT NULL,
        z DOUBLE PRECISION  NOT NULL,
        name TEXT NOT NULL
    );
""")


2025-12-20T18:00:27+0100 INFO	26834	__main__	1025727644.py	sqllite_connection	5	Opening connection-interface to ('/home/fenke/repos/EDCompanion/data/eddb_systems.db',)
2025-12-20T18:00:27+0100 INFO	26834	__main__	3862603891.py	sqllite_connection_interface	13	Using isolation_level=None
2025-12-20T18:00:27+0100 INFO	26834	__main__	3862603891.py	sqllite_connection_interface	38	Returning connection-interface
2025-12-20T18:00:27+0100 INFO	26834	__main__	3862603891.py	close	17	Closing connection


In [ ]:
print(f"Adding indexes ...")
with sqllite_connection(systems_databasefile) as conn:
    conn.execute(f"""
        CREATE INDEX IF NOT EXISTS systems_x_idx ON systems (x);
        CREATE INDEX IF NOT EXISTS systems_y_idx ON systems (y);
        CREATE INDEX IF NOT EXISTS systems_z_idx ON systems (z); 
        CREATE INDEX IF NOT EXISTS systems_name_idx ON systems (name);
        CREATE INDEX IF NOT EXISTS systems_id64_idx ON eddb.systems (id64)
    """)

In [ ]:
with sqllite_connection(systems_databasefile) as conn:

    print("Removing duplicates by id")
    conn.execute("""
            DELETE FROM systems
            WHERE rowid NOT IN (
                SELECT MIN(rowid)
                FROM systems
                GROUP BY x, z, id64
            );                 

        """
    )

    print(f"Adding unique index on system id64 ...")
    conn.execute(f"""
        DROP INDEX systems_id64_idx ;
        CREATE UNIQUE INDEX IF NOT EXISTS systems_id64_idx ON eddb.systems (id64)
    """)

#### Update systems

In [ ]:
data_dump_file = os.path.join(eddb_config['local_dumps'], 'systems_1day.json'),
data_dump_file

('/home/fenke/repos/EDCompanion/data/systems_1day.json',)

In [ ]:
import pandas as pd


In [ ]:
pd.read_json(data_dump_file, lines=True, orient='records').head(5)

ValueError: Invalid file path or buffer object type: <class 'tuple'>

In [ ]:
def get_coordinates_from_item(item):
    coords = item.get('coords')
    return {k:coords[k] for k in ['x','y','z']}

conn = sqllite_connection_interface(systems_databasefile)
query = """
    INSERT INTO systems (id64, x, y, z, name) 
    VALUES (:id,:x,:y,:z,:name) 
    ON CONFLICT DO NOTHING

"""


def process_data(datachunk):

    return conn.executemany(query, [
                dict(id=item.get("id64"), **get_coordinates_from_item(item), name=item.get('name'))
                for item in datachunk
            ]
    )

dbfile_process(
    data_dump_file,
    process_data,
)

In [ ]:
SQLite doesn’t have a dedicated BULK INSERT command like some other databases, but you can achieve high‑performance bulk imports by combining:

Transactions (wrap all inserts in a single transaction to avoid per‑row commits)
Prepared statements (reuse compiled SQL with parameters)
Batch execution (loop over data and bind parameters)

Here’s a complete, runnable Python example using sqlite3 that efficiently imports thousands of rows:
Python

import sqlite3
import csv
import os

def bulk_import_csv(db_path, table_name, csv_path):
    # Validate file existence
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    # Connect to SQLite database (creates file if it doesn't exist)
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # Example: create table if not exists
        cursor.execute(f"""
            CREATE TABLE IF NOT EXISTS {table_name} (
                id INTEGER PRIMARY KEY,
                name TEXT NOT NULL,
                age INTEGER NOT NULL
            )
        """)

        # Open CSV and read data
        with open(csv_path, newline='', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            headers = next(reader, None)  # Skip header row if present

            # Start transaction for speed
            conn.execute("BEGIN TRANSACTION;")

            # Use parameterized insert for safety and performance
            cursor.executemany(
                f"INSERT INTO {table_name} (name, age) VALUES (?, ?)",
                reader
            )

            # Commit transaction
            conn.commit()

        print(f"Data imported successfully into '{table_name}'.")

    except sqlite3.Error as e:
        conn.rollback()
        print(f"SQLite error: {e}")
    finally:
        conn.close()


# Example usage:
if __name__ == "__main__":
    bulk_import_csv("people.db", "persons", "people.csv")

Why this is fast:

Single transaction: Without it, SQLite commits after each row, which is slow.
executemany(): Sends multiple parameter sets to the same prepared statement.
Parameterized queries: Prevent SQL injection and reuse compiled SQL.


CLI Alternative (No Python)
If you already have a CSV file and table ready, you can use the SQLite shell:
Bashsqlite3 mydb.db <<EOF
.mode csv
.import mydata.csv mytable
EOF


Note: .import is fastest when the table is empty and has no indexes.


If you want, I can also give you a C or C++ version using sqlite3_prepare_v2() and sqlite3_bind_*() for maximum performance in native code.
Do you want me to prepare that?


### Star types

In [ ]:
main_sequence = {s:c for s, c in zip('OBAFGKMN', range(9))}
#print(json.dumps(classifications, indent=2))


In [ ]:

star_types={}
star_types_file = os.path.join(eddb_config['local_data_folder'], 'star_types.json')
print(f"File with star types: {star_types_file}")


File with star types: /home/fenke/repos/EDCompanion/data/star_types.json


In [ ]:

if not os.path.exists(star_types_file):
    print("Extracting star types")
    skipped = 0
    updated = 0

    for item in dbfilereader(os.path.join(eddb_config['local_dumps'], eddb_config['systems_1month'])):
        main_star = item.get('mainStar')

        if main_star and not star_types.get(main_star):
            print(f"updating with {main_star}")
            updated += 1
            star_types[main_star] = len(star_types)
        else:
            skipped += 1


    print(f"Updated: {updated}, skipped {skipped}")


    main_star_types = {}
    for name in set(star_types.keys()):
        first, *rest = name.split(' ')
        if len(first) == 1 and first in main_sequence:
            main_star_types[name] = len(main_star_types)

    for name in star_types:
        if name not in main_star_types:
            main_star_types[name] = len(main_star_types)

    try:
        with open(star_types_file,'wt') as jsonfile:
            json.dump(main_star_types, jsonfile, indent=3)
    except:
        pass

    star_types = main_star_types.copy()
    
else:
    with open(star_types_file,'rt') as jsonfile:
        star_types.update(json.load(jsonfile))

#print(json.dumps(main_star_types, indent=2))


In [ ]:
main_sequence

{'O': 0, 'B': 1, 'A': 2, 'F': 3, 'G': 4, 'K': 5, 'M': 6, 'N': 7}

In [ ]:
os.path.join(eddb_config['local_dumps'], eddb_config['systems_1week'])

'/home/fenke/repos/EDCompanion/data/systems_1week.json.gz'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()